# 使用案例:单次推荐 / Use case: one recommendation

最小可用示例:给定**一个**厨房配料清单,拿到 Top-5 食谱。用的是已训练好的最终模型
`lda_model.pkl`(K=6),**无需重训**。(从 `model/` 文件夹运行。)

A minimal example: given **one** pantry, get the Top-5 recipes. It uses the already-trained
final model `lda_model.pkl` (K=6) — **no retraining**. (Run from the `model/` folder.)

## 1 · 载入最终模型 / Load the final model

In [1]:
import pandas as pd
import recipe_recommender as rr

df = pd.read_csv("recipes_clean.csv")
model = rr.load_model("lda_model.pkl")           # the final model (K=6)
rr._STATE = {"model": model, "df_id": id(df)}    # let recommend() reuse it (no retrain)

print(f"Loaded final model (K={model.best_k}) and {len(df):,} recipes")

g++ not available, if using conda: `conda install gxx`


Loaded final model (K=6) and 53,573 recipes


## 2 · 我的厨房 / My pantry tonight

今晚冰箱里有这些 / Tonight I have these on hand:

In [2]:
pantry = ["chicken", "garlic", "onion", "olive oil", "tomato",
          "rice", "salt", "black pepper"]

recs = rr.recommend(pantry, df, top_n=5)         # one call -> Top-5
pd.DataFrame(recs)[["recipe_name", "coverage", "missing_ingredients",
                    "predicted_rating", "posterior_uncertainty", "score"]]

[filter_candidates] threshold=0.7: 25 candidate recipes


,recipe_name,coverage,missing_ingredients,predicted_rating,posterior_uncertainty,score
0,solo sweet onion rice,4/5 ingredients,[chicken stock],4.088,0.1446,1.5461
1,authentic italian spaghetti sauce,5/7 ingredients,"[basil, crushed red pepper flake]",4.345,0.1131,1.5081
2,italian dipping oil for bread,3/4 ingredients,[basil],4.724,0.0929,1.3411
3,broiled chicken with oil lemon and garlic sauce,5/7 ingredients,"[lemon juice, parsley]",4.660,0.3577,1.3030
4,tasty bbq tomatoes,3/4 ingredients,[basil],4.539,0.0891,1.2963


## 3 · 读懂第一条推荐 / Reading the top pick

- **coverage** —— 你已有 / 食谱所需 的配料数 / how many of the recipe's ingredients you already have.
- **missing_ingredients** —— 还缺什么 / what you'd still need to buy.
- **flavor_tags** —— 后验给出的两个风味主题 / the two flavor topics from the posterior.
- **predicted_rating** —— 贝叶斯收缩后的评分 / Bayesian-shrunk rating.
- **posterior_uncertainty** —— 风味对齐在 MCMC 样本上的标准差(模型有多确定)/ how (un)sure the model is.

In [3]:
top = recs[0]
print("Top pick:", top["recipe_name"])
print("  coverage         :", top["coverage"])
print("  missing          :", top["missing_ingredients"] or "nothing - you can make it now!")
print("  flavor tags      :", "  |  ".join(top["flavor_tags"]))
print("  predicted rating :", top["predicted_rating"], "/ 5")
print("  uncertainty (std):", top["posterior_uncertainty"])
print("  score            :", top["score"])

Top pick: solo sweet onion rice
  coverage         : 4/5 ingredients
  missing          : ['chicken stock']
  flavor tags      : onion / salt / garlic clove  |  olive oil / garlic clove / black pepper
  predicted rating : 4.088 / 5
  uncertainty (std): 0.1446
  score            : 1.5461


## 4 · 加点约束 / Add a constraint

同样一次调用,只是带上选项:**只要素食**,而且**必须用到番茄**。
Same single call, just with options: **vegetarian only**, and it **must use the tomato**.

In [4]:
veg = rr.recommend(pantry, df, diet="vegetarian", must_use=["tomato"], top_n=5)
pd.DataFrame(veg)[["recipe_name", "coverage", "missing_ingredients", "predicted_rating"]]

[filter_candidates] threshold=0.7: 25 candidate recipes
[filters] diet=vegetarian, must_use=['tomato']: 25 -> 8 recipes


,recipe_name,coverage,missing_ingredients,predicted_rating
0,authentic italian spaghetti sauce,5/7 ingredients,"[basil, crushed red pepper flake]",4.345
1,tasty bbq tomatoes,3/4 ingredients,[basil],4.539
2,stewed tomatoes and garbanzo beans,5/7 ingredients,"[garlic clove, garbanzo bean]",4.543
3,best marinara sauce,4/5 ingredients,[basil leaf],3.020
4,chef flower s simple avocado dip,5/7 ingredients,"[avocado, lemon juice]",4.817


## 就这么简单 / That's it

一次 `import`,一次 `recommend()` 调用——其余选项(`top_n` / `exclude` / `must_use` /
`diet` / `diversity`)和字段含义见 README 的 Step 5。

One import, one `recommend()` call. For the other options (`top_n` / `exclude` /
`must_use` / `diet` / `diversity`) and field meanings, see Step 5 in the README.